In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**First we must frame the problem:**

How does the business/users expect to use and benefit from this model:
Managers want to free up time spent organising and working out the priority of a ticket. Using a model to predict what priority a ticket would be frees up support worker time to focus on actually closing tickets.

What kind of training supervision will the model need:
This model will need supervised training as the data fed to the algorithm will include the desired solution

Is this a regression or classification task:
This will be a classification task, based on input features we will need to classify the priority of a ticket.

Batch learning or Online learning:
Batch learning because the data is stable with infrequent model updates.


**Select a performance measure:**

Because we are predicting a classification outcome we should start with F1 Score, the F1 score is a number that tells how good the classifier is at finding the positive in each class while balancing precision (How many tickets did the model predict the priority of correctly) and recall (How many of each priority was caught)

# GET THE DATA

In [ ]:
import kagglehub
import pandas
from pathlib import Path
from kagglehub import KaggleDatasetAdapter


# Load the latest version of the dataset
def load_ticket_data():
    file_path = Path("/kaggle/input/customer-support-ticket-dataset/customer_support_tickets.csv")
    if not file_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)    
        df = kagglehub.load_dataset(
            KaggleDatasetAdapter.PANDAS,
            "suraj520/customer-support-ticket-dataset",
            file_path,
        )
    return pandas.read_csv(file_path)

ticket = load_ticket_data()



Let's take a look at the data before we proceed

- Use the .head, .info and .describe methods to get a general overview of the data
- Can see if any attributes are missing data
- Can see if any columns require preprocessing (e.g. the chest pain type column may require one hot encoding)

In [ ]:
ticket.head(10)

In [ ]:
ticket.info()

In [ ]:
ticket.describe()

Let's clean up some of these unessecary columns before we get to deep into the training process

In [ ]:
ticket.drop(columns=['Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender', 'Product Purchased', 'Resolution',
                    'Ticket Status', 'First Response Time', 'Time to Resolution', 'Customer Satisfaction Rating'], inplace=True,  errors='ignore')
ticket.head()

# CREATE A TEST SET

Set apart a random 20% of the dataset for later use.

In [ ]:
import numpy as np

def shuffle_and_split_data(data, test_ratio):
    shuffled_indices = np.random.permutation(len(data)) #len(data) gets the number of rows in the dataset,
                                                            #np.random.permutation() returns a random ordering of integers (shuffles the row numbers)
    test_set_size = int(len(data) * test_ratio) #Gets the number of rows that will be in the test set.
    test_indices = shuffled_indices[:test_set_size] #Gets the first test_set_size number of shuffled indices
    train_indices = shuffled_indices[test_set_size:] #Gets the rest.
    return data.iloc[train_indices], data.iloc[test_indices] # Selects the rows 


train_set, test_set = shuffle_and_split_data(ticket, 0.2)

len(train_set)

In [ ]:
len(test_set)

This isn't a perfect solution as each time this is run a different test set will be generated, over time the machine learning algorithm
will see the whole dataset, a better way of doing this is to create a hash of each instance's identifier and use that to see if its lower than or
equal to the 20% of the maximum hash value.

In [ ]:
from zlib import crc32

def is_id_in_test_set(identifier, test_ratio):
    return crc32(np.int64(identifier)) < test_ratio * 2**32 #crc32 computes the crc32 hash of the input number
                                                            #produces a 32-bit unsinged integer hash value - basically a pseudo-random number
def split_data_with_id_hash(data, test_ratio, id_column):
    ids = data[id_column]
    in_test_set = ids.apply(lambda id_: is_id_in_test_set(id_, test_ratio))
    return data.loc[~in_test_set], data.loc[in_test_set]
    # df.loc Selects rows meeting logical condition, ~ means select all that aren't in the test_set 

train_set, test_set = split_data_with_id_hash(ticket, 0.2, "Ticket ID") #Ticket has a unique identifier so use that

len(train_set)

This has used random sampling, I think a samf assumption can be made that the Ticket Type heavily impacts the ticket priority, so it might be worth doing statified sampling where we will seperate the dataset into srata (ticket types) to make sure that the right number of instances are sampled from each stratum

First we have to convert the ticket type column into discrete numbers

In [ ]:
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

le = LabelEncoder()
ticket_type_col = le.fit_transform(ticket["Ticket Type"])

#ticket_type_col.value_counts().sort_index().plot.bar(rot=0, grid=True)
#plt.xlabel("Ticket Categories")
#plt.ylabel("Number of Tickets")
#plt.show()



list(zip(le.classes_, range(len(le.classes_))))



The following code generates 10 different stratified splits of the same dataset.

- Multiple splits let you evaluate your model more robustly through cross-validation. Each split serves as a different train/test combination.

So what is occuring here?
The dataset is being divided into two subsets, Training and Test but the StratifiedShuffleSplit ensures that the split is randomized, but reproducible with random_state, and each subset retains the same class distribution.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

def split_data_with_strat():
    splitter = StratifiedShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
    strat_splits=[]
    for train_index, test_index in splitter.split(ticket, ticket["Ticket Type"]):
        strat_train_set_n = ticket.iloc[train_index]
        strat_test_set_n = ticket.iloc[test_index]
        strat_splits.append([strat_train_set_n, strat_test_set_n])
    strat_train_set, strat_test_set = strat_splits[0]
    return strat_train_set, strat_test_set # Just use the first split for now


# CLEAN THE DATA
This data has plenty of missing features these should be set to the median value.

All the columns that are being dropped do not serve the purpose of classifying the priority of support tickets, they may be added back if something else needs to be done with the data. Notice how the data is split after dropping the columns but before any data transforming takes place.

In [ ]:
strat_train_set, strat_test_set = split_data_with_strat()
#strat_train_set.Drop(["Ticket ID"], inplace=True)
#strat_test_set = strat_test_set.Drop(["Ticket ID"])

for set_ in (strat_train_set, strat_test_set):
    set_.drop("Ticket ID", axis=1, inplace=True)
strat_train_set.info()

Categorical attributes, such as priority, type and channel can be encoded, but we must be careful how we encode each one.
For ordered categories such as the ticket priority, these values can simply be encoded, but for none ordered like the ticket type and channel these should be OneHotEncoded. 

The following code demonstrates how this may be done, it doesn't actually transform the data.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

priority_order = [["Low", "Medium", "High", "Critical"]]

ticket_priority_cat = ticket[["Ticket Priority"]]
ordinal_encoder = OrdinalEncoder(categories=priority_order)
ticket_priority_cat_encoded = ordinal_encoder.fit_transform(ticket_priority_cat)

ordinal_encoder.categories_

In [ ]:
ticket_priority_cat_encoded[:8]
ticket.info()

# MAKE A PIPELINE

Many data transformation steps need to be executed in a specific order, so a pipeline needs to be created

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

one_hot_cat_attribs = ["Ticket Channel", "Ticket Type" ]

one_hot_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)

preprocesser = ColumnTransformer([
    ("text_desc", TfidfVectorizer(max_features=1000, stop_words="english"), "Ticket Description"),
    ("text_sub", TfidfVectorizer(max_features=1000, stop_words="english"), "Ticket Subject"),
    ("hot", one_hot_pipeline, one_hot_cat_attribs)
])

Now our pipeline has been built we *simply* train the model

- x_train is the training feature set, includes all columns except the target column
- y_train is the training target set, includes only the target column
- x_test is the test feature set, is the inputs the model will use to make predictions
- y_test is the test target set, is the labels for those inputs

In [ ]:
from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline([
    ("preprocessing", preprocesser),
    ("classifier", RandomForestClassifier())
])

priority_order = [["Low", "Medium", "High", "Critical"]]
label_encoder = OrdinalEncoder(categories=priority_order)

x_train = strat_train_set.drop("Ticket Priority", axis=1)
y_train = strat_train_set["Ticket Priority"]
x_test = strat_test_set.drop("Ticket Priority", axis=1)
y_test = strat_test_set["Ticket Priority"]

y_train_encoded = label_encoder.fit_transform(y_train.values.reshape(-1, 1))
y_test_encoded = label_encoder.transform(y_test.values.reshape(-1, 1))

pipeline.fit(x_train, y_train_encoded.ravel())
predictions = pipeline.predict(x_test)

We have the predictions made by the model now, now we need to get the corresponding labels in the y_test set to see how well it has done

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y_test_encoded, predictions)
#print(accuracy)
x_train.head()

Precision - How many predicted as X are actually X

Recall - How many of the actualy X were correctly predicted

F1-score - Mean of precision and recall

Support - Number of true samples in the class

In [ ]:
print(classification_report(y_test_encoded, predictions))

Each cell [i][j] is the count of samples actual=i predicted=j

In [ ]:
print(confusion_matrix(y_test_encoded, predictions))

# IMPROVE MODEL

Seems like the model predictions are quite weak, the quickest and easiest thing to test is if a more complex model will help.


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import FunctionTransformer

to_dense_transformer = FunctionTransformer(lambda x: x.toarray() if hasattr(x, "toarray") else x)

pipeline = Pipeline([
    ("preprocessing", preprocesser),
    ("to_dense", to_dense_transformer),
    ("gnb", MultinomialNB())
])

priority_order = [["Low", "Medium", "High", "Critical"]]
label_encoder = OrdinalEncoder(categories=priority_order)

x_train = strat_train_set.drop("Ticket Priority", axis=1)
y_train = strat_train_set["Ticket Priority"]
x_test = strat_test_set.drop("Ticket Priority", axis=1)
y_test = strat_test_set["Ticket Priority"]

y_train_encoded = label_encoder.fit_transform(y_train.values.reshape(-1, 1))
y_test_encoded = label_encoder.transform(y_test.values.reshape(-1, 1))

pipeline.fit(x_train, y_train_encoded.ravel())
predictions = pipeline.predict(x_test)

In [ ]:
accuracy = accuracy_score(y_test_encoded, predictions)
print(accuracy)

In [ ]:
print(classification_report(y_test_encoded, predictions))

Still not great, it may be worth fine tuning some parameters but otherwise it is probably a lack of data that is causing the issue

In [ ]:
from sklearn import svm
from sklearn.preprocessing import StandardScaler

def to_dense_transform(x):
    return x.toarray() if hasattr(x, "toarray") else x

pipeline = Pipeline([
    ("preprocessing", preprocesser),
    ("to_dense", FunctionTransformer(to_dense_transform)),
    ("scaler", StandardScaler(with_mean=False)),
    ("svm", svm.SVC(probability=True))
])

priority_order = [["Low", "Medium", "High", "Critical"]]
label_encoder = OrdinalEncoder(categories=priority_order)

x_train = strat_train_set.drop("Ticket Priority", axis=1)
y_train = strat_train_set["Ticket Priority"]
x_test = strat_test_set.drop("Ticket Priority", axis=1)
y_test = strat_test_set["Ticket Priority"]

y_train_encoded = label_encoder.fit_transform(y_train.values.reshape(-1, 1))
y_test_encoded = label_encoder.transform(y_test.values.reshape(-1, 1))

pipeline.fit(x_train, y_train_encoded.ravel())
predictions = pipeline.predict(x_test)

In [ ]:
import numpy as np
unique, counts = np.unique(y_train_encoded, return_counts=True)
print(dict(zip(unique, counts)))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test_encoded, predictions)
disp = ConfusionMatrixDisplay(confusion_matrix = cm)
disp.plot()

After trying with multiple Models, we can see that a consistent ~25% accuracy is achieved across all of them. Based on the fact that we only have 4 classes, this is pretty much random guessing, so it's clear that the model is not at fault, it's either the data or the way we are processing it. It seems the data isn't best suited for support ticket priority classification, this should have been reviewed before picking the dataset :( so that's a mistake on my part.

# DEPLOY MODEL
Despite the model not being incredibly accurate I still want to deploy it and comsume it in the accompanying C# application I have build

In [ ]:
import joblib

joblib.dump(pipeline, "ticket_classifier_model.pkl")

Sample prediction

In [ ]:
example = pd.DataFrame([{
    "Date of Purchase": "2025-09-24T00:00:00+10:00",
    "Ticket Type": 5,
    "Ticket Subject": "I dont know",
    "Ticket Description": "This is a go pro and it's broken",
    "Ticket Channel": 0
}])

print(pipeline.predict(example))

# THROW THE WHOLE THING OUT AND USE A PRE-MADE MODEL

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# Encode the labels again
y_train_encoded = label_encoder.fit_transform(y_train.values.reshape(-1, 1))
y_test_encoded = label_encoder.transform(y_test.values.reshape(-1, 1)) 

# Just using the description for now
train_dataset = Dataset.from_dict({
    "text": x_train["Ticket Description"].tolist(),
    "label": y_train_encoded.tolist()
})

val_dataset = Dataset.from_dict({
    "text": x_test["Ticket Description"].tolist(),
    "label": y_test_encoded.tolist()
})

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def preprocess(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)


train_dataset = train_dataset.map(preprocess, batched=True)
val_dataset = val_dataset.map(preprocess, batched=True)

model= AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
)


def compute_metrics(pred):
    labels = pred.label_ids                    # true labels
    preds = np.argmax(pred.predictions, axis=1)  # predicted labels
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.save_model("./ticket_priority_model")
tokenizer.save_pretrained("./ticket_priority_model")
print("MADE IT")